# 09 · Integration, inner products & Sobolev norms

`omnibias-fields` turns the closed-form field values and derivatives into
first-class *functional-analysis* operators: a definite integral over a box
domain, weighted inner products, and L² / Sobolev (Hᵏ) norms. The nodes and
weights come from a shared NumPy `QuadratureSpec`, so torch and jax integrate
**bit-identically**.

We use a small analytic field (`notebooks/_fields.py`) whose integrals and
norms have exact closed forms, so every number below can be checked.

In [ ]:
import sys

import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
from _style import set_style, PRIMARY, ACCENT, WARM
from _fields import make_field, Poly
set_style()

torch.set_default_dtype(torch.float64)

from omnibias.fields.torch import _ops_dispatch as dispatch
from omnibias.fields.torch.ops.integral import integrate, quadrature_nodes
from omnibias.fields.torch.ops.norms import inner_product, l2_norm, sobolev_norm
from omnibias.fields._core.quadrature import gauss_legendre, monte_carlo

print("ready")

## Definite integration: quadrature vs Monte-Carlo

Integrate `u(x) = x⁴` on `[0, 1]` (exact value `1/5`). Gauss–Legendre with a
handful of nodes is *exact* for polynomials; Monte-Carlo converges at the usual
`1/√N` rate.

In [ ]:
poly = make_field(("x",), {"u": (Poly((0.0, 0.0, 0.0, 0.0, 1.0)),)}, dispatch)  # x^4
exact = 1.0 / 5.0


def integral_with(rule):
    nodes = quadrature_nodes(rule, like=torch.zeros(1))
    return float(integrate(poly(nodes), "u", rule=rule))


gl = integral_with(gauss_legendre([(0.0, 1.0)], 8))
Ns = [50, 200, 1000, 5000, 20000]
mc_errs = [abs(integral_with(monte_carlo([(0.0, 1.0)], N, seed=0)) - exact) for N in Ns]

print(f"Gauss-Legendre (8 nodes): {gl:.15f}   |error| = {abs(gl - exact):.2e}")
print(f"exact:                    {exact:.15f}")

## Inner products & Sobolev norms

For `u(x) = x²` and `v(x) = x` on `[0, 1]`, every quantity has a closed form:

- `‖u‖²_L²  = ∫ x⁴ = 1/5`
- `‖u‖²_H¹  = ∫ x⁴ + ∫ (2x)² = 1/5 + 4/3`
- `‖u‖²_H²  = 1/5 + 4/3 + ∫ 2² = 1/5 + 4/3 + 4`
- `⟨u, v⟩   = ∫ x³ = 1/4`

In [ ]:
uv = make_field(
    ("x",),
    {"u": (Poly((0.0, 0.0, 1.0)),), "v": (Poly((0.0, 1.0)),)},  # u=x^2, v=x
    dispatch,
)
rule = gauss_legendre([(0.0, 1.0)], 16)
st = uv(quadrature_nodes(rule, like=torch.zeros(1)))

l2 = float(l2_norm(st, "u", rule=rule))
h1 = float(sobolev_norm(st, "u", rule=rule, k=1))
h2 = float(sobolev_norm(st, "u", rule=rule, k=2))
ip = float(inner_product(st, "u", "v", rule=rule))

print(f"||u||_L2 = {l2:.12f}   exact {np.sqrt(1/5):.12f}")
print(f"||u||_H1 = {h1:.12f}   exact {np.sqrt(1/5 + 4/3):.12f}")
print(f"||u||_H2 = {h2:.12f}   exact {np.sqrt(1/5 + 4/3 + 4):.12f}")
print(f"<u, v>   = {ip:.12f}   exact {1/4:.12f}")

In [ ]:
fig, ax = plt.subplots()
ax.loglog(Ns, mc_errs, "o-", color=ACCENT, label="Monte-Carlo |error|")
ref = [mc_errs[0] * (Ns[0] / n) ** 0.5 for n in Ns]
ax.loglog(Ns, ref, "--", color=WARM, label=r"$1/\sqrt{N}$ reference")
ax.axhline(max(abs(gl - exact), 1e-17), color=PRIMARY,
           label="Gauss-Legendre (8 nodes)")
ax.set_xlabel("samples N")
ax.set_ylabel("absolute error")
ax.set_title(r"Integrating $x^4$ on $[0,1]$: quadrature vs Monte-Carlo")
ax.legend()
plt.tight_layout()
plt.show()

## Takeaway

Integration, inner products, and Sobolev norms are closed-form operators built
on the field's exact derivatives — no autograd, and bit-identical across
backends. They are the building blocks for variational/energy losses and
Galerkin projections.

Next: **[10 · Geometry on the sphere](10_geometry_sphere_laplace_beltrami.ipynb)**.